### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='**Why do parrots talk?**  \nParrots are among the world’s most vocal “talkers” because their brains, bodies, and social lives are all wired to learn and reproduce sounds. Here’s the science (and a little bit of fun) behind the chatter:\n\n| What makes them talk | How it happens | Why it matters |\n|----------------------|----------------|----------------|\n| **Vocal learning** | Parrots possess a highly flexible syrinx (the bird “voice box”) that can produce a wide range of sounds. Their brains have a large *song system* that allows them to store, rehearse, and fine‑tune vocal patterns. | The ability to copy sounds gives parrots a huge advantage in the wild—mimicking predators, warning calls, or even the sounds of other species helps them stay safe and communicate. |\n| **Social bonding** | In the wild, parrots live in flocks and use vocalizations to keep track of one another, coordinate foraging, and signal danger. Mimicking a fellow bird’s call is a way of saying “

In [5]:
from langchain.tools import tool

@tool       #it is a decorator. when we use it a top of a function, that function actually become a tool in langchain
def get_weather(location:str)->str:
    """Get the weather at a location"""  #it is called a docstring,when we bind it with tool the llm will be able to find the functionlities of the tool 
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_82a73761-d07d-464c-8348-15ed057069d5', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 127, 'total_tokens': 162, 'completion_time': 0.039141186, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.010765404, 'prompt_tokens_details': None, 'queue_time': 1.311511267, 'total_time': 0.04990659}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a16910-5e20-750f-900e-b9fe6a852467-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_82a73761-d07d-464c-8348-15ed057069d5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."